# INC-001 Tabular profiling: participants, device metadata, wellness survey

Read-only profiling notebook. It **proposes** rules and decides nothing. Standard library only; writes no files;
prints counts, aggregates and participant/date identifiers, never raw measurement rows.
Cell ids (`tab-NN`) are used as references in `reports/INC-001/profiling-findings.md`.
Study window per data dictionary: 2026-01-08 to 2026-02-04 (28 days), fixed local timezone UTC-05:00.

In [1]:
import csv, re, time, collections, statistics
from datetime import date, timedelta
from pathlib import Path

T0 = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "participants").is_dir())
DATA = ROOT / "data"
WIN_START, WIN_END = date(2026, 1, 8), date(2026, 2, 4)
WINDOW_DAYS = [WIN_START + timedelta(days=i) for i in range(28)]
assert WINDOW_DAYS[-1] == WIN_END

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        return reader.fieldnames, list(reader)

def blank(v):
    return v is None or v.strip() == ""

def show(label, value, examples=None):
    extra = f"  examples={sorted(examples)[:5]}" if examples else ""
    print(f"{label}: {value}{extra}")

pcols, participants = read_csv(DATA / "participants" / "participants.csv")
dcols, devices = read_csv(DATA / "health_summaries" / "device_metadata.csv")
scols, survey = read_csv(DATA / "health_summaries" / "wellness_survey.csv")
print("columns participants:", pcols)
print("columns device_metadata:", dcols)
print("columns wellness_survey:", scols)

columns participants: ['participant_id', 'device_id', 'age', 'height_cm', 'gender', 'chronotype', 'max_heart_rate']
columns device_metadata: ['participant_id', 'device_id', 'wear_site', 'calibration_date', 'firmware_version', 'device_label']
columns wellness_survey: ['participant_id', 'date', 'fatigue_score', 'stress_score', 'readiness_score', 'sleep_quality_score']


## 1. Row counts and key uniqueness

In [2]:
print("rows participants:", len(participants), "| device_metadata:", len(devices), "| wellness_survey:", len(survey))
pid_counts = collections.Counter(r["participant_id"] for r in participants)
show("participants duplicate participant_id", sum(1 for c in pid_counts.values() if c > 1), [k for k, c in pid_counts.items() if c > 1])
did_counts = collections.Counter(r["device_id"] for r in participants)
show("participants duplicate device_id", sum(1 for c in did_counts.values() if c > 1), [k for k, c in did_counts.items() if c > 1])
dm_counts = collections.Counter(r["participant_id"] for r in devices)
show("device_metadata duplicate participant_id", sum(1 for c in dm_counts.values() if c > 1), [k for k, c in dm_counts.items() if c > 1])
dm_dev = collections.Counter(r["device_id"] for r in devices)
show("device_metadata duplicate device_id", sum(1 for c in dm_dev.values() if c > 1))
key_counts = collections.Counter((r["participant_id"], r["date"]) for r in survey)
dups = [k for k, c in key_counts.items() if c > 1]
show("survey duplicate (participant_id, date) keys", len(dups), [f"{p}@{d}" for p, d in dups])
show("survey exact-duplicate rows", len(survey) - len({tuple(r.items()) for r in survey}))
show("participant_id format ^P\\d{3}$ violations", sum(1 for r in participants if not re.fullmatch(r"P\d{3}", r["participant_id"])))
show("device_id format FBT-NNNN-XXXXXXXX violations", sum(1 for r in participants if not re.fullmatch(r"FBT-\d{4}-[0-9A-F]{8}", r["device_id"])))
show("survey rows with leading/trailing whitespace in any field", sum(1 for r in survey if any(v != v.strip() for v in r.values() if v)))

rows participants: 50 | device_metadata: 50 | wellness_survey: 1414
participants duplicate participant_id: 0
participants duplicate device_id: 0
device_metadata duplicate participant_id: 0
device_metadata duplicate device_id: 0
survey duplicate (participant_id, date) keys: 0
survey exact-duplicate rows: 0
participant_id format ^P\d{3}$ violations: 0
device_id format FBT-NNNN-XXXXXXXX violations: 0
survey rows with leading/trailing whitespace in any field: 0


## 2. Nulls and blanks (including `firmware_version` vs `device_label`)

In [3]:
for name, rows, cols in (("participants", participants, pcols), ("device_metadata", devices, dcols), ("wellness_survey", survey, scols)):
    counts = {c: sum(1 for r in rows if blank(r[c])) for c in cols}
    print(name, "blank counts per column:", {c: n for c, n in counts.items() if n} or "none")

FW_RE = re.compile(r"fw(\d+(?:\.\d+)*)")
blank_fw = [r for r in devices if blank(r["firmware_version"])]
show("device_metadata rows with blank firmware_version", len(blank_fw), [r["participant_id"] for r in blank_fw])
parsed_ok = [r for r in blank_fw if FW_RE.search(r["device_label"] or "")]
show("  of which device_label carries a parseable fw<version>", len(parsed_ok))
both = [r for r in devices if not blank(r["firmware_version"]) and FW_RE.search(r["device_label"] or "")]
mismatch = [r for r in both if FW_RE.search(r["device_label"]).group(1) != r["firmware_version"].strip()]
show("rows with both values where firmware_version != fw in device_label", len(mismatch), [r["participant_id"] for r in mismatch])
no_label_fw = [r for r in devices if not FW_RE.search(r["device_label"] or "")]
show("rows whose device_label has no parseable fw<version>", len(no_label_fw), [r["participant_id"] for r in no_label_fw])
SN_RE = re.compile(r"SN:(FBT-\d{4}-[0-9A-F]{8})")
dev_by_pid = {r["participant_id"]: r["device_id"] for r in participants}
sn_mismatch = [r for r in devices if (SN_RE.search(r["device_label"] or "") or [None, None])[1] != r["device_id"]]
show("device_label SN differs from device_id column", len(sn_mismatch), [r["participant_id"] for r in sn_mismatch])
print("firmware versions in firmware_version column:", dict(collections.Counter(r["firmware_version"] for r in devices)))
print("firmware versions parsed from device_label:", dict(collections.Counter((FW_RE.search(r["device_label"]) or [None, "?"])[1] for r in devices)))

participants blank counts per column: none
device_metadata blank counts per column: {'firmware_version': 15}
wellness_survey blank counts per column: none
device_metadata rows with blank firmware_version: 15  examples=['P006', 'P008', 'P015', 'P016', 'P021']
  of which device_label carries a parseable fw<version>: 15
rows with both values where firmware_version != fw in device_label: 0
rows whose device_label has no parseable fw<version>: 0
device_label SN differs from device_id column: 0
firmware versions in firmware_version column: {'2.2.0': 23, '2.2.1': 12, '': 15}
firmware versions parsed from device_label: {'2.2.0': 23, '2.2.1': 12, '2.1.8': 15}


## 3. Value domains and ranges

In [4]:
def to_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None

print("gender:", dict(collections.Counter(r["gender"] for r in participants)))
print("chronotype:", dict(collections.Counter(r["chronotype"] for r in participants)))
print("wear_site:", dict(collections.Counter(r["wear_site"] for r in devices)))
for col in ("age", "height_cm", "max_heart_rate"):
    vals = [to_int(r[col]) for r in participants]
    bad = [r["participant_id"] for r, v in zip(participants, vals) if v is None]
    good = [v for v in vals if v is not None]
    print(f"{col}: non-integer={len(bad)} min={min(good)} max={max(good)} median={statistics.median(good)}")
offsets = [to_int(r["max_heart_rate"]) - (220 - to_int(r["age"])) for r in participants]
print("max_heart_rate minus (220-age): min", min(offsets), "max", max(offsets), "(dictionary: 'plus individual offset')")
cal = []
for r in devices:
    try:
        cal.append((date.fromisoformat(r["calibration_date"]), r["participant_id"]))
    except ValueError:
        cal.append((None, r["participant_id"]))
show("calibration_date unparseable", sum(1 for d, _ in cal if d is None))
valid_cal = [d for d, _ in cal if d]
print("calibration_date range:", min(valid_cal), "to", max(valid_cal), "| after window start:", sum(1 for d in valid_cal if d >= WIN_START))

SCORE_RANGES = {"fatigue_score": (1, 5), "stress_score": (1, 5), "readiness_score": (0, 10), "sleep_quality_score": (1, 5)}
for col, (lo, hi) in SCORE_RANGES.items():
    vals = [to_int(r[col]) for r in survey]
    nonint = [(r["participant_id"], r["date"]) for r, v in zip(survey, vals) if v is None and not blank(r[col])]
    blanks = sum(1 for r in survey if blank(r[col]))
    good = [v for v in vals if v is not None]
    out = [(r["participant_id"], r["date"]) for r, v in zip(survey, vals) if v is not None and not lo <= v <= hi]
    print(f"{col}: documented {lo}-{hi}; observed {min(good)}-{max(good)}; blank={blanks}; non-integer={len(nonint)}; out-of-range={len(out)}",
          sorted(f"{p}@{d}" for p, d in out)[:5] if out else "")

gender: {'male': 38, 'female': 12}
chronotype: {'B': 22, 'A': 28}
wear_site: {'wrist_nondominant': 20, 'wrist_dominant': 30}
age: non-integer=0 min=25 max=59 median=30.0
height_cm: non-integer=0 min=163 max=195 median=179.5
max_heart_rate: non-integer=0 min=157 max=203 median=188.0
max_heart_rate minus (220-age): min -28 max 37 (dictionary: 'plus individual offset')
calibration_date unparseable: 0
calibration_date range: 2025-12-04 to 2026-01-07 | after window start: 0
fatigue_score: documented 1-5; observed 1-5; blank=0; non-integer=0; out-of-range=0 
stress_score: documented 1-5; observed 1-5; blank=0; non-integer=0; out-of-range=0 
readiness_score: documented 0-10; observed 0-10; blank=0; non-integer=0; out-of-range=0 
sleep_quality_score: documented 1-5; observed 2-5; blank=0; non-integer=0; out-of-range=0 


## 4. Participant / device linkage and orphans

In [5]:
P = {r["participant_id"] for r in participants}
D = {r["participant_id"] for r in devices}
S = {r["participant_id"] for r in survey}
folders = {p.name for p in (DATA / "wearable_events").iterdir() if p.is_dir()}
show("participants without device_metadata row", len(P - D), P - D)
show("device_metadata rows without participant (orphans)", len(D - P), D - P)
show("survey participant_ids not in participants (orphans)", len(S - P), S - P)
show("participants without any survey row", len(P - S), P - S)
show("wearable_events folders not in participants", len(folders - P), folders - P)
show("participants without wearable_events folder", len(P - folders), P - folders)
missing_files = [(p, f) for p in sorted(folders) for f in ("sleep.json", "steps.json", "heart_rate.json") if not (DATA / "wearable_events" / p / f).is_file()]
show("folders missing one of the three JSON files", len(missing_files), [f"{p}/{f}" for p, f in missing_files])
dev_mismatch = [r["participant_id"] for r in devices if dev_by_pid.get(r["participant_id"]) not in (None, r["device_id"])]
show("device_id differs between participants.csv and device_metadata.csv", len(dev_mismatch), dev_mismatch)
show("device_ids in metadata not in participants.csv", len({r["device_id"] for r in devices} - set(did_counts)))

participants without device_metadata row: 0
device_metadata rows without participant (orphans): 0
survey participant_ids not in participants (orphans): 1  examples=['P999']
participants without any survey row: 0
wearable_events folders not in participants: 0
participants without wearable_events folder: 0
folders missing one of the three JSON files: 0
device_id differs between participants.csv and device_metadata.csv: 0
device_ids in metadata not in participants.csv: 0


## 5. Survey dates vs study window and per-participant day coverage

The survey grain is tested here: is it exactly one row per participant per day?

In [6]:
parsed = collections.defaultdict(set)
bad_dates, outside = [], []
for r in survey:
    try:
        d = date.fromisoformat(r["date"])
    except ValueError:
        bad_dates.append((r["participant_id"], r["date"]))
        continue
    parsed[r["participant_id"]].add(d)
    if not WIN_START <= d <= WIN_END:
        outside.append((r["participant_id"], d.isoformat()))
show("survey dates not ISO yyyy-mm-dd", len(bad_dates), [f"{p}@{d}" for p, d in bad_dates])
show("survey rows outside study window", len(outside), [f"{p}@{d}" for p, d in outside])
cover = {p: len(parsed[p] & set(WINDOW_DAYS)) for p in sorted(P | S)}
print("distinct in-window survey days per participant (histogram days->participants):", dict(sorted(collections.Counter(cover.values()).items())))
short = [p for p, n in cover.items() if n < 28]
show("participants with <28 in-window survey days", len(short), short)
missing_days = sum(28 - cover[p] for p in P)
show("missing participant-days in survey among enrolled participants (of 50 x 28 = 1400)", missing_days)
show("survey rows belonging to non-enrolled participant_ids", sum(1 for r in survey if r["participant_id"] not in P))
by_day = collections.Counter(d for p in parsed for d in parsed[p] if WIN_START <= d <= WIN_END)
print("survey rows per study day: min", min(by_day.values()), "max", max(by_day.values()), "days with no rows:", [d.isoformat() for d in WINDOW_DAYS if d not in by_day])
distinct_keys = len(key_counts)
print(f"survey rows={len(survey)} distinct (participant_id,date)={distinct_keys} -> daily grain per participant (at most one row per participant-day): {distinct_keys == len(survey)}")
print("one row per participant per day over full window (28 per participant):", all(cover[p] == 28 for p in P) and distinct_keys == len(survey))

survey dates not ISO yyyy-mm-dd: 0
survey rows outside study window: 0
distinct in-window survey days per participant (histogram days->participants): {14: 1, 28: 50}
participants with <28 in-window survey days: 1  examples=['P999']
missing participant-days in survey among enrolled participants (of 50 x 28 = 1400): 0
survey rows belonging to non-enrolled participant_ids: 14
survey rows per study day: min 50 max 51 days with no rows: []
survey rows=1414 distinct (participant_id,date)=1414 -> daily grain per participant (at most one row per participant-day): True
one row per participant per day over full window (28 per participant): True


## Runtime

In [7]:
print(f"total notebook runtime: {time.perf_counter() - T0:.2f} s")

total notebook runtime: 48.02 s
